In [ ]:
import json
import copy
from torch.utils.data import Dataset
import datas

class RaftLogDataSet(Dataset):
    def __init__(self, datafilesName):
        self.datafilesName = datafilesName
        self.serversOrignDatas = []
        for i,fileName in enumerate( self.datafilesName):
           with open(fileName, 'r') as f:
                lines = f.readlines()
                tmp = []
                for line in lines:
                    if len(line) < 2:continue
                    js_data = json.loads(line)
                    if('role' in js_data):
                        if(js_data['role'] == 'action' and js_data['action'] == 'change_state'):
                            js_data['me'] = i
                    tmp.append(js_data)
                self.serversOrignDatas.append(tmp)
                f.close()
        print("load datafiles done...")      
        self.all_servers_datasets = []
        
       
        for i,serverOrignData in enumerate(self.serversOrignDatas) :
            flattend_jsons = []
            for js_data in serverOrignData:
                if('coredump' in js_data):continue
                if ('role' not in js_data) or (line is None) or ('timestamp' not in js_data):
                    continue
                if('action' in js_data):
                    js_data['me'] = i
                keys,values = datas.flattenJson(js_data)
                
                if(len(keys) != len(values)):continue
                tmp_js = dict(zip(keys,values))
                if(tmp_js['role'] == 'state'):
                    tmp_js.pop('raft_state.ip_index')
                    tmp_js.pop('raft_state.nextIndex')
                    lo_rx_str = 'system_state.network.process.lo:.RX Bytes'
                    lo_tx_str = 'system_state.network.process.lo:.TX Bytes'
                    tmp_js.pop(lo_rx_str)
                    tmp_js.pop(lo_tx_str)
                flattend_jsons.append(tmp_js)
            self.all_servers_datasets.append(flattend_jsons)
        print("flattend_jsons ...")
        combined_actionTuples = []
        '''
        combined_actionTuples :[ (timestamp,[(serverId,actionJson,index) ... ]) ... ]
        '''
        def timestampIsExistInTuples(combined_actionTuples,timeStamp) -> int:
            for i in range(len(combined_actionTuples)):
                if combined_actionTuples[i][0] == timeStamp:
                    return i
            return -1
        
        def getPrevStateJsonInServerByIndex(serverId,actionIndex):
            serverDatas = self.all_servers_datasets[serverId]
            for i in range(actionIndex,-1,-1):
                if serverDatas[i]['role'] == 'state':
                    return serverDatas[i]
            return None

        def getNextStateJsonInServerByIndex(serverId,actionIndex):
            serverDatas = self.all_servers_datasets[serverId]
            for i in range(actionIndex,len(serverDatas)):
                if serverDatas[i]['role'] == 'state':
                    return serverDatas[i]
            return None
        
        for serverId in range(len(self.all_servers_datasets)):
            serverDatas = self.all_servers_datasets[serverId]
            for i  in range(len(serverDatas)):
                json_data = serverDatas[i]
                if(json_data['role'] != 'action'):continue
                timestamp = json_data['timestamp']
                idx = timestampIsExistInTuples(combined_actionTuples,timestamp)
                tuple_ = (serverId,json_data,i)
                if idx == -1:
                    combined_actionTuples.append((timestamp,[tuple_]))
                else:   
                    combined_actionTuples[idx][1].append(tuple_)
                    
        self.train_data_each_batch_by_logicOrder = []
        for tp_ in combined_actionTuples:
            # tp_ : (timestamp,[(serverId,actionJson,index) ... ])
            actionTupleList = tp_[-1]
            grouped_by_serverId = {}
            for actionTuple in actionTupleList:
                # actionTuple : (serverId,actionJson,index)
                serverId = actionTuple[0]
                if serverId not in grouped_by_serverId:
                    grouped_by_serverId[serverId] = []
                grouped_by_serverId[serverId].append(actionTuple)
                
            batch_tuple = {}
            for serverId in grouped_by_serverId:
                batch_tuple[serverId] = {
                    'prevStateJson' : None,
                    'actions' : [],
                    'nextStateJson':None
                }
                actionTuples = grouped_by_serverId[serverId]
                prevStateJson = None
                nextStateJson = None
                # actionTuple: (serverId,actionJson,index)
                for actionTuple in actionTuples:
                    actionJson = actionTuple[1]
                    index = actionTuple[2]
                    if prevStateJson is None:
                        prevStateJson = getPrevStateJsonInServerByIndex(serverId,index)
                    if nextStateJson is None:
                        nextStateJson = getNextStateJsonInServerByIndex(serverId,index)
                    batch_tuple[serverId]['actions'].append(actionJson)
                batch_tuple[serverId]['prevStateJson'] = prevStateJson
                batch_tuple[serverId]['nextStateJson'] = nextStateJson
            self.train_data_each_batch_by_logicOrder.append(batch_tuple)
          

    def __getitem__(self, index):
        
        if(index >= len(self.train_data_each_batch_by_logicOrder) or index < 0):
            raise IndexError("Index out of range")
        batch_tuple = self.train_data_each_batch_by_logicOrder[index]
        once_prevStateJson = []
        once_actions = []
        once_nextStateJson = []
        for serverId in batch_tuple:
            item = batch_tuple[serverId]
            prevStateJson = item['prevStateJson']
            nextStateJson = item['nextStateJson']
            actions = item['actions']
            once_prevStateJson.append(prevStateJson)
            once_nextStateJson.append(nextStateJson)
            once_actions.append(actions)  
        return (once_prevStateJson,once_actions,once_nextStateJson  )
    
    def __len__(self):
        return len(self.train_data_each_batch_by_logicOrder)


In [ ]:
datafilesName0= ['/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host0/system_runtime.data0',
                 '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host1/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host2/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host3/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host4/system_runtime.data0',
                ]
datafilesName1 = ['/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host0/system_runtime.data1',
                 '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host1/system_runtime.data1',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host2/system_runtime.data1',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host3/system_runtime.data1',
                     '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host4/system_runtime.data1',
                ]
datafilesName2 = ['/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host0/system_runtime.data2',
                    '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host1/system_runtime.data2',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host2/system_runtime.data2',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host3/system_runtime.data2',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host4/system_runtime.data2',
                ]
datafilesName3 = ['/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host0/system_runtime.data3',
                    '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host1/system_runtime.data3',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host2/system_runtime.data3',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host3/system_runtime.data3',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host4/system_runtime.data3',
]
datafilesName4 = ['/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host0/system_runtime.data4',
                    '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host1/system_runtime.data4',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host2/system_runtime.data4',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host3/system_runtime.data4',
                        '/home/cdy/code/projects/cRaft/.data/mutil_state_raft_datas/host4/system_runtime.data4',
]
datafilesNames = [datafilesName0,datafilesName1,datafilesName2,datafilesName3,datafilesName4]
datasets = []
for datafilesName in datafilesNames:
    dataset = RaftLogDataSet(datafilesName)
    datasets.append(dataset)


In [ ]:
len(datas.Action)

In [ ]:
import torch
import random
import math
out_labels = []
def getBatchDatas(batch_size:int,dataset):

    def handleOriginFlattendJson(js):
        lo_rx_str = 'system_state.network.process.lo:.RX Bytes'
        lo_tx_str = 'system_state.network.process.lo:.TX Bytes'
        if(lo_rx_str in js):js.pop(lo_rx_str)
        if(lo_tx_str in js):js.pop(lo_tx_str)
        if('timestamp' in js):js.pop('timestamp')
        if('role' in js):js.pop('role')
        if('raft_state.ip_index' in js):js.pop('raft_state.ip_index')
        if('raft_state.nextIndex' in js):js.pop('raft_state.nextIndex')
        if('ip_index' in js):js.pop('ip_index')
        if('nextIndex' in js):js.pop('nextIndex')
        # if('raft_state.commitIndex' in js):js.pop('raft_state.commitIndex')
        # if('raft_state.lastLogIndex' in js):js.pop('raft_state.lastLogIndex')
        if('raft_state.logsize' in js):js.pop('raft_state.logsize')
        if('raft_state.snapShotIndex' in js):js.pop('raft_state.snapShotIndex')
        if('raft_state.snapShotTerm' in js):js.pop('raft_state.snapShotTerm')
        # if('raft_state.term' in js):js.pop('raft_state.term')
        for key in js:
            if js[key] is None:
                js[key] = 0
            if(key.startswith('system_state')):
                js[key] = math.log10(1 + js[key])
        return js
    batchs = []
    one_batch_x = []
    one_batch_y = []
    batch_size =batch_size

    keep_servers_nearest_state = {0:None,1:None,2:None,3:None,4:None}
    for time_step in range(len(dataset)):
        if(len(one_batch_x) == batch_size):
            batchs.append((one_batch_x,one_batch_y))
            one_batch_x = []
            one_batch_y = []
        once_prevStateJson,once_actions,once_nextStateJson = dataset[time_step]
        once_nextStateJson = copy.deepcopy(once_nextStateJson)
        once_prevStateJson = copy.deepcopy(once_prevStateJson)
        once_actions = copy.deepcopy(once_actions)
        for prevJson in once_prevStateJson:
            if(prevJson is None):continue
            prevJson = handleOriginFlattendJson(prevJson)
            serverid = prevJson['raft_state.me']
            ens_rx_str = 'system_state.network.process.ens33:.RX Bytes'
            ens_tx_str = 'system_state.network.process.ens33:.TX Bytes'
        
            dis_ens_rx_bytes = abs(prevJson[ens_rx_str] - prevJson[ens_rx_str])
            dis_ens_tx_bytes = abs(prevJson[ens_tx_str] - prevJson[ens_tx_str])
        
            prevJson[ens_rx_str] = dis_ens_rx_bytes
            prevJson[ens_tx_str] = dis_ens_tx_bytes
            keep_servers_nearest_state[serverid] = list(prevJson.values())
            
        is_state_changed = False
        label_list = [[0] * len(datas.Action) for _ in range(5)]
        for servers_actionlist in once_actions:
            for action_json in servers_actionlist:
                # is_ok ,success, is voted
                if(action_json is None):continue
                action_json = handleOriginFlattendJson(action_json)
                cur_action = action_json['action']
                me = -1
                if(action_json['action'] == datas.Action.CHANGE_STATE.value):
                    me = action_json['me']
                else:
                    if('me' in action_json):
                        me = action_json['me']
                    elif ('id' in action_json):
                        me = action_json['id']
                    else:
                        print(action_json)
                if(cur_action == datas.Action.CHANGE_STATE.value):
                    to_state = datas.state_to_int[action_json['to']].value
                    if(to_state == datas.State.LEADER.value):
                        label_list[me][datas.Action.TO_L.value] = 1
                        label_list[me][datas.Action.CHANGE_STATE.value] = 1
                        for i in range(5):
                            if i == me:continue
                            label_list[i][datas.Action.TO_F.value] = 1
                            label_list[i][datas.Action.CHANGE_STATE.value] = 1
                        is_state_changed = True
                        break
                else:
                    is_ok = False
                    peer = -1
                    if('is_ok' in action_json):
                        is_ok = action_json['is_ok']
                    elif ('success' in action_json):
                        is_ok = action_json['success']
                    elif ('is_voted' in action_json):
                        is_ok = action_json['is_voted']
                    if('peer' not in action_json):
                        peer = action_json['to']
                    else:
                        peer = action_json['peer']
                    action_peer = None
                    if(peer == 0):
                        action_peer = datas.Action.CALL_0.value
                    elif(peer == 1):
                        action_peer = datas.Action.CALL_1.value
                    elif(peer == 2):    
                        action_peer = datas.Action.CALL_2.value
                    elif(peer == 3):
                        action_peer = datas.Action.CALL_3.value
                    elif(peer == 4):
                        action_peer = datas.Action.CALL_4.value
                    if(action_peer is None):continue
                    if(is_ok):
                        label_list[me][action_json['action']] = 1
                        label_list[me][action_peer]  = 1
                    # out_labels.append(label_list)
                    # print(label_list)
            if(is_state_changed):break
        x = list(keep_servers_nearest_state.values())
        y = label_list
        is_contain_none = False
        for item in x:
            if(item is None):
                is_contain_none = True
                break
        if(is_contain_none):continue
            
        one_batch_x.append(x)
        one_batch_y.append(y)
    if(len(one_batch_x) > 0):
        batchs.append((one_batch_x,one_batch_y))
    return batchs   

In [ ]:
import action_model
out_test_batchs =[]
tmp_label = []

avg_loss = []
losses = []
test_loss = []
def train(num_eopchs,batch_size = 1500):
    
    num_eopchs = num_eopchs
    num_servers = len(datasets)
    print("num_servers:",num_servers)
    action_num = len(datas.Action)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  
    embed_dim = 512
    num_heads  = 2
    lr = 1e-4
        
    input_width =  101
    model =action_model.CenterActionNetwork(num_servers,action_num,input_width,embed_dim,num_heads).to(device)
    #初始化模型参数
    def init_weights(m):
        if type(m) == torch.nn.Linear:
            torch.nn.init.xavier_uniform(m.weight)
            m.bias.data.fill_(0.01)
    model.apply(init_weights)
   
    pos_weight = torch.ones([action_num])
    # pos_weight[datas.Action.TO_L.value] = 10
    # pos_weight[datas.Action.TO_F.value] = 10
    # pos_weight[datas.Action.CHANGE_STATE.value] = 10
    pos_weight[datas.Action.APPEND_ENTRIES.value] = 10
    pos_weight[datas.Action.REQUEST_VOTE.value] =  10
    # loss_fn =  torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
    loss_fn =  torch.nn.BCEWithLogitsLoss().to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    ####################
   
    min_test_loss = 1000000
    train_ratio = 0.5
    train_batches = []
    test_batches = []
    for dataset in datasets:
        batchs = getBatchDatas(batch_size,dataset)
        train_len = int(len(batchs) * train_ratio)
        train_batch = batchs[:train_len]
        test_batch = batchs[train_len:]
        train_batches.append(train_batch)
        test_batches.extend(test_batch)
        out_test_batchs.extend(test_batch)
    for epoch in range(num_eopchs):
        model.train()
        for i ,train_batch in enumerate( train_batches):
            random.shuffle(train_batch)
            for j,(batch_x,batch_y) in enumerate(train_batch):
                print(f"epoch:{epoch},train_batch:",i,"/",len(train_batches),"batch:",j,"/",len(train_batch))
                tmp_label.extend(batch_y)
                input_x = torch.tensor(batch_x,dtype=torch.float).to(device)
                y_label =  torch.tensor(batch_y,dtype=torch.float).to(device)
                output ,_= model(input_x)
                output = output.view(-1,action_num)
                y_label = y_label.view(-1,action_num)
                loss =  loss_fn(output, y_label)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                print(epoch,loss.item())
                losses.append(loss.item())
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            # # # eval model
        model.eval()
        temp_loss = []
        for batch_x,batch_y in test_batches:
            input_x = torch.tensor(batch_x,dtype=torch.float).to(device)
            y_label =  torch.tensor(batch_y,dtype=torch.float).to(device)
            output ,_= model(input_x)
            output = output.view(-1,action_num)
            probs = torch.sigmoid(output)
            predictions = (probs > 0.5).int()  
            y_label = y_label.view(-1,action_num)
            loss = loss_fn(output, y_label)
            test_loss.append(loss.item())
            temp_loss.append(loss.item())
        if(len(temp_loss) == 0):continue
        tmp_avg_loss = sum(temp_loss) / len(temp_loss)
        temp_loss = []
        if(epoch >800 and tmp_avg_loss < min_test_loss):
            min_test_loss = tmp_avg_loss
            torch.save(model, f'{epoch}-{min_test_loss}-model.pth')
                
        print("train done")
train(2000)

In [ ]:
len(tmp_label[0])

In [ ]:
model = torch.load('/home/cdy/code/projects/cRaft/src/train/model/best-action-model.pth').eval()
tmp = []   
attion_out = [] 
random.shuffle(out_test_batchs)
for (x,y) in out_test_batchs:
    input_x = torch.tensor(x,dtype=torch.float).to('cuda')
    out ,att_out= model(input_x)
    attion_out.append(att_out)
    probs = torch.sigmoid(out)
    predictions = (probs > 0.5).int()  
    tmp.extend(predictions.cpu().detach().numpy())


In [ ]:
def plot_aggregated_heatmap(data, custom_labels):
    """
    绘制所有预测结果的平均分布热图，并自定义 x 轴标签。
    
    Args:
        data (list of 2D arrays): 每个元素是 [num_servers, num_actions] 的二维数组。
        custom_labels (list of str): 自定义 x 轴标签。
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 计算所有预测结果的平均值
    avg_prediction = np.mean(data, axis=0)

    # 绘制聚合热图
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(avg_prediction, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, linewidths=0.5)

    # 设置自定义的 x 轴标签
    ax.set_xticks(range(len(custom_labels)))
    ax.set_xticklabels(custom_labels, rotation=45, ha="right")

    plt.title("Average Action Distribution Across Predictions", fontsize=12)
    plt.xlabel("Significant Actions Feature", fontsize=12)
    plt.ylabel("Hosts", fontsize=12)
    plt.tight_layout()
    dpis = [300,600,800,1200]
    for dpi in  dpis:
        plt.savefig(f'./action-prediction-{dpi}-heatmap.png', dpi=dpi,bbox_inches="tight")
    plt.show()



In [ ]:
new_t = []
for tmp_ in tmp:
    data_array = np.array(tmp_)

    # 删除第 0、2、3、4 列
    columns_to_delete = [0, 3, 4,5,6]
    filtered_data = np.delete(data_array, columns_to_delete, axis=1)
    new_t.append(filtered_data.tolist())

In [ ]:

# 调用函数
# plot_animated_heatmap(tmp[0:100])
custom_labels = ["","Receive RPC", "Call RPC", "Call 0","Call 1","Call 2","Call 3","Call 4"]  # 自定义的 x 轴标签
plot_aggregated_heatmap(new_t,custom_labels)


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def tsne_visualization(data, random_state=42, perplexity=30, learning_rate=200, n_iter=1000):
    """
    用 t-SNE 将高维数据降到 2D 并绘制散点图。

    Parameters:
    - data (numpy.ndarray): 形状为 (n_samples, n_features) 的高维数据数组。
    - random_state (int): t-SNE 随机种子，确保结果可复现。
    - perplexity (float): t-SNE 的困惑度，建议根据样本大小调整（如 5-50）。
    - learning_rate (float): t-SNE 的学习率，通常在 10-1000 之间。
    - n_iter (int): 最大迭代次数。

    Returns:
    - tsne_result (numpy.ndarray): 降维后的 2D 数据，形状为 (n_samples, 2)。
    """
    if not isinstance(data, np.ndarray):
        raise ValueError("数据必须是 numpy.ndarray 格式。")
    if data.shape[1] != 512:
        raise ValueError("输入数据的每个样本维度必须为 512。")

    # 使用 t-SNE 进行降维
    tsne = TSNE(n_components=2, perplexity=perplexity, learning_rate=learning_rate,
                n_iter=n_iter, random_state=random_state, init='random')
    print("正在降维，请稍等...")
    tsne_result = tsne.fit_transform(data)
    print("降维完成！")

    # 绘制降维后的 2D 数据
    plt.figure(figsize=(10, 8))
    plt.scatter(tsne_result[:, 0], tsne_result[:, 1], s=1, alpha=0.6)
    plt.title("t-SNE Visualization of High-Dimensional Data", fontsize=14)
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(alpha=0.3)
    plt.show()

    return tsne_result

random.shuffle(hiddens)
# to numpy
hiddens = np.array(hiddens)
tsne_result = tsne_visualization(hiddens[:])


In [ ]:
t = []
for b in tmp:
    for i in range(len(b)):
       t.append(b[i])
random.shuffle(t)
for a in t:
    print(a)

In [ ]:
# calculate the last test loss
last_test_loss = []
model.eval()
tmp = []
for batch_x,batch_y in test_batchs:
    input_x = torch.tensor(batch_x,dtype=torch.float).to(device)
    y_label =  torch.tensor(batch_y,dtype=torch.float).to(device)
    output = model(input_x)
    probs = torch.sigmoid(output)  # shape: (batch_size, 5, 16)

# 使用阈值（比如 0.5）来决定每个主机选择了哪些动作
    predictions = (probs > 0.5).int()  # 将概率大于 0.5 的动作标记为 1，其他为 0
 
    tmp.append(predictions)
    loss = loss_fn(output, y_label)
    last_test_loss.append(loss.item())
print("last test loss:",sum(last_test_loss)/len(last_test_loss))

In [ ]:
for b in tmp:
    for i in range(len(b)):
        print(b[i])
    print("===============")

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
def readLossFileToList(fileName):
    with open(fileName, 'r') as f:
        lines = f.readlines()
        losses = []
        for line in lines:
            if len(line) < 2:continue
            losses.append(float(line))
        f.close()
    return losses

def average_k_elements(lst, k):
    if k <= 0:
        raise ValueError("K must be a positive integer greater than zero.")
    averages = []
    for i in range(0, len(lst), k):
        group = lst[i:i+k]  
        group_average = sum(group) / len(group)  
        averages.append(group_average)  
    return averages

#归一化函数
def normalize(lst):
    s = sum(lst)
    return list(map(lambda x: x/s, lst))

k = 10
# losses_list = readLossFileToList('../out/loss.txt')
# test_loss_list = readLossFileToList('../out/test_loss.txt')
# last_test_loss_list = readLossFileToList('../out/last_test_loss.txt')
losses_list = losses
test_loss_list = test_loss
last_test_loss_list = last_test_loss
losses_result = average_k_elements(losses_list[:],1000)


plt.plot(normalize(losses_result),label='Training loss',marker='+',markersize=4,linewidth=1,linestyle='--')
test_result = average_k_elements(test_loss_list[:],180)
plt.plot(normalize(test_result),label='Validation  loss',marker='^',markersize=3,linewidth=1,linestyle='--')
plt.plot([0.0067] * len(losses_result),label='Final test avg_loss')


# 添加网格线
plt.grid(linewidth=0.5,alpha = 0.7,linestyle='--')

plt.title('Raft cluster(5 hosts) action network training loss',fontsize=12)
#设置x轴标签
plt.xlabel('Time step',fontsize=12)
#设置y轴标签
plt.ylabel('Average loss change',fontsize=12)
plt.legend(fontsize=12)
plt.xticks(np.arange(0, 38, 2),fontsize = 7)  # 从0到10，间隔为2
plt.yticks(np.arange(0.0, 0.15, 0.01),fontsize = 8)  # 从-
x = [0.0,36]
y = [0.145,0.0060]
plt.plot(x,y,c='gray',linewidth=0.8,linestyle='--')
dpis = [300,500,800,1200]
for dpi in dpis:
    plt.savefig(f'../out/action_loss-{dpi}.png',dpi=dpi)
# result = average_k_elements(test_loss_list[:],k)
# plt.plot(result,label='test loss')
# result = average_k_elements(last_test_loss_list[:],k)
# plt.plot(result,label='last test loss')





In [ ]:
f = open('../tmp_save/action-loss.txt','w')
for item in losses:
    f.write(str(item) + '\n')
f.close()
f = open('../tmp_save/action-test_loss.txt','w')
for item in test_loss:
    f.write(str(item) + '\n')
f.close()
f = open('../out/last_test_loss.txt','w')

